# [클래스, 상속, 파일 입출력, 예외 처리 - '종합 미니 프로젝트: 스마트 은행 시스템']

#### 14일차: 2주차 종합 복습 프로젝트 설계
이번 프로젝트의 핵심은 각 개념이 서로 어떻게 '협력'하는지 이해하는 것.

1. 커스텀 예외 정의 (예외 처리): 먼저 우리 은행만의 규칙을 어겼을 때 발생할 에러들을 정의.
2. 클래스 구조 (상속과 다형성)
- 부모(BankAccount): 공통 기능(입출력)을 가짐
- 자식(SavingsAccount): 이자율 개념을 추가
- 자식(CheckingAccount): 수수료 개념을 추가
3. 데이터 영속성 (파일 입출력): 거래가 발생할 때마다 log.txt 파일에 기록하여 나중에 확인할 수 있게 함

In [1]:
# [2주차 결산 - 종합 은행 시스템 프로젝트]

import datetime

# 1.커스텀 예외 (예외 처리)
class BankError(Exception):
    pass

class InsufficientFundsError(BankError):
    """잔액 부족 에러"""
    pass

# 2.부모 클래스 (캡슐화 적용)
class BankAccount:
    def __init__(self, account_num, owner, balance=0):
        self.account_num = account_num
        self.owner = owner
        self.__balance = balance # private 변수 보호

    @property
    def balance(self):
        return self.__balance

    def _log_transaction(self, msg):
        # 3.파일 저장 (파일 입출력)
        with open("bank_log.txt", "a", encoding="utf-8") as f:
            now = datetime.datetime.now
            f.write(f"[{now}] {self.owner}({self.account_num}): {msg}\n")

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("입금액은 0보다 커야 합니다.")
        self.__balance += amount
        self._log_transaction(f"{amount}원 입금 (잔액: {self.__balance}원)")
        print(f"✅ {amount}원 입금 완료")

    def withdraw(self, amount):
        if amount > self.__balance:
            raise InsufficientFundsError(f"잔액 부족 (현재: {self.__balance}원)")
        self.__balance -= amount
        self._log_transaction(f"{amount}원 출금 (잔액: {self.__balance}원)")
        print(f"✅ {amount}원 출금 완료")

# 4.자식 클래스 (상속 및 다형성)
class SavingsAccount(BankAccount):
    """이자율이 있는 정기적금"""
    def add_interest(self, rate):
        interest = self.balance * rate
        self.deposit(interest)
        print(f"💰 이자 {interest}원이 추가되었습니다.")

# 5.실행 및 테스트
try:
    bob_acc = SavingsAccount("123-456", "Bob", 10000)
    bob_acc.deposit(5000)
    bob_acc.withdraw(3000)
    bob_acc.add_interest(0.2) # 2% 이자 추가

    # 예외 발생 테스트 (잔액 부족)
    bob_acc.withdraw(20000)

except InterruptedError as e:
    print(f"❌ 은행 오류: {e}")
except ValueError as e:
    print(f"❌ 입력 오류: {e}")
except Exception as e:
    print(f"❌ 알 수없는 오류: {e}")
finally:
    print("\n--- 거래를 종료합니다. 로그 파일(bank_log.txt)을 확인하세요. ---")

✅ 5000원 입금 완료
✅ 3000원 출금 완료
✅ 2400.0원 입금 완료
💰 이자 2400.0원이 추가되었습니다.
❌ 알 수없는 오류: 잔액 부족 (현재: 14400.0원)

--- 거래를 종료합니다. 로그 파일(bank_log.txt)을 확인하세요. ---


**[핵심 요약]**
1. 클래스: 우리가 만드는 '통장 공장'
2. 상속: 기본 통장 공장에서 '이자 주는 통장'이라는 특별판을 만드는 것. 기본 기능은 그대로 쓰고 새로운 기술만 넣음
3. 파일 저장: 거래가 끝낼 때마다 통장에 "도장"을 쾅! 찍어서 기록을 남기는 것과 같음
4. 예외 처리: "돈도 없는데 돈 뽑아줘!"라고 떼쓰는 상황을 대비해서 '경과 알람'을 맞춰놓은 것